# Bundling OpenMM simulations for NanoVer

This notebook demonstrates how to take an OpenMM molecular simulation setup and bundle it into a compact format that can be used in NanoVer for convenience.

## OpenMM simulation setup

First we set up an OpenMM simulation (example adapted from [OpenMM documentation](https://docs.openmm.org/latest/userguide/application/02_running_sims.html)).

In [7]:
from openmm import unit, LangevinMiddleIntegrator
from openmm.app import ForceField, PME, HBonds, PDBFile, Simulation

pdb = PDBFile("../systems/input.pdb")
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1*unit.nanometer,
    constraints=HBonds,
)

integrator = LangevinMiddleIntegrator(
    300*unit.kelvin,
    1/unit.picosecond,
    0.004*unit.picoseconds,
)

simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()

## Bundling the simulation

Next we bundle the constructed simulation into a single file; a compressed archive of OpenMM's own serialization of the simulation components.

In [2]:
from nanover.openmm import bundle_openmm_simulation

bundle_openmm_simulation(simulation, outfile="villin-headpiece.openmm.zip")

## Unbundling the simulation

Unbundling is equally simple:

In [3]:
from nanover.openmm import unbundle_openmm_simulation

simulation_copy = unbundle_openmm_simulation("villin-headpiece.openmm.zip")

In [4]:
print(simulation.topology, "\n", simulation_copy.topology)

<Topology; 1 chains, 2798 residues, 8867 atoms, 6111 bonds> 
 <Topology; 1 chains, 2798 residues, 8867 atoms, 6111 bonds>


## NanoVer simulation from OpenMM bundle

Additionally one can create a NanoVer ready simulation directly from a bundle and serve it over the network as usual.

In [5]:
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_bundle_path("villin-headpiece.openmm.zip")
omm_sim.load()

In [6]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(omm_sim, port=0, name="openmm example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "openmm example" (ws://localhost:57500), discoverable on all interfaces on port 54545
Available simulations:
[0]: "villin-headpiece.openmm"
Switched to [0]: "villin-headpiece.openmm"
Switched to [0]: "villin-headpiece.openmm"
